In [10]:
import numpy as np
import yaml
from pathlib import Path
from deploy_real.g1_arm_IK import *

In [ ]:
record_dir = Path("records")
test_file = "traj_1766477199.npz"
data = np.load(record_dir / test_file)

In [7]:
print(data["note"])
print(data["traj"].shape)
print(data["traj"][500])

cols=[q(12)|pL(3)|qL(4)|pR(3)|qR(4)]
(1099, 26)
[ 0.36180377  1.86366689  1.13655996  1.03041565  1.02996027 -0.48910981
  0.17027187 -1.72448206 -0.89123088  0.853912   -0.76921946  0.58561254
  0.10828982  0.59183977  0.49233108  0.6858673   0.46239827  0.40037793
  0.39429861  0.15300785 -0.58794788  0.43402429  0.6673428  -0.41320668
  0.33259805 -0.52277372]


In [11]:
class Config: pass

def load_cfg(path="deploy_real/configs/config_high_level.yaml"):
    with open(path, 'r') as f:
        d = yaml.safe_load(f)
    cfg = Config()
    for k, v in d.items():
        setattr(cfg, k, np.array(v) if isinstance(v, list) else v)
    cfg.kps_record = cfg.kps_play * 0
    cfg.kds_record = cfg.kds_play * 0
    cfg.replay_transition_duration = 1.5
    return cfg

cfg = load_cfg()

In [48]:
print(cfg.action_joints)
print(type(cfg.action_joints))

[15 16 17 18 19 21 22 23 24 25 26 28]
<class 'numpy.ndarray'>


In [21]:
ik = G1_29_ArmIK()

[G1_29_ArmIK] >>> Loading cached robot model: g1_29_model_cache.pkl


In [44]:
def FK(q):
        '''
        forward kinematics
        param:  q: arm motor joint state (n-joints)
        return: tuple: tuple(left_hand_f, right_hand_f) of (pin.SE3)
        '''
        pin.forwardKinematics(ik.reduced_robot.model, ik.reduced_robot.data, q)
        pin.updateFramePlacements(ik.reduced_robot.model, ik.reduced_robot.data)
        d = ik.reduced_robot.data
        return d.oMf[ik.L_hand_id], d.oMf[ik.R_hand_id]

def IK(q, poseL: pin.SE3, poseR: pin.SE3):
        '''
        inverse kinematics
        param:  poseL: left  hand end-effector pose
                poseR: right hand end-effector pose
        return: q   : arm motor joint state (n-joints)
        '''
        q_now = q
        q_cmd, _ = ik.solve_ik(poseL.homogeneous, poseR.homogeneous, current_lr_arm_motor_q=q_now)
        return q_cmd

def get_action_q(q_full):
    return q_full[cfg.action_joints]

In [50]:
q_record_dir = Path("deploy_real/records")
qfilel = "single_1.npz"
qfiler = "single_2.npz"
qdata_l = np.load(q_record_dir / qfilel)['q']
qdata_r = np.load(q_record_dir / qfiler)['q']
ql = get_action_q(qdata_l)
qr = get_action_q(qdata_r)
qf = np.concatenate([ql[:6], qr[6:]])
print("Input joint angles:\n", list(qf))
poseL, poseR = FK(qf)
print("Left hand pose:\n", poseL)
print("Right hand pose:\n", poseR)

Input joint angles:
 [-0.06300107389688492, 1.6363739967346191, 0.06309694796800613, 1.433660864830017, -0.06913699209690094, -0.01609145849943161, -0.023728765547275543, -1.5769442319869995, -0.06358829885721207, 1.4105912446975708, 0.04851214215159416, 0.04398690164089203]
Left hand pose:
   R =
  0.152118  -0.189341   0.970057
  0.986127 -0.0368914  -0.161839
 0.0664295   0.981218   0.181103
  p = 0.0588338  0.653798  0.335101

Right hand pose:
   R =
  0.171154   0.126411   0.977101
 -0.984961  0.0457494   0.166612
-0.0236403  -0.990922    0.13234
  p = 0.0632459 -0.654936  0.298884



In [40]:
print(repr(poseL.rotation))
print(type(poseL.rotation))
print(repr(poseL.translation))
print(type(poseL.translation))
print(poseL.np)
print(poseL.homogeneous)

array([[ 0.15212, -0.18934,  0.97006],
       [ 0.98613, -0.03689, -0.16184],
       [ 0.06643,  0.98122,  0.1811 ]])
<class 'numpy.ndarray'>
array([0.05883, 0.6538 , 0.3351 ])
<class 'numpy.ndarray'>
[[ 0.15212 -0.18934  0.97006  0.05883]
 [ 0.98613 -0.03689 -0.16184  0.6538 ]
 [ 0.06643  0.98122  0.1811   0.3351 ]
 [ 0.       0.       0.       1.     ]]
[[ 0.15212 -0.18934  0.97006  0.05883]
 [ 0.98613 -0.03689 -0.16184  0.6538 ]
 [ 0.06643  0.98122  0.1811   0.3351 ]
 [ 0.       0.       0.       1.     ]]


In [56]:
quit_q = get_action_q(np.array([-0.089,-0.018,-0.006, 0.534,-0.422, 0.001,
                -0.09 ,-0.008, 0.013, 0.531,-0.429, 0.004,
                0, 0.   , 0.   ,
                0.053, 0.486, 0.173, 1.249, 0.167,0.121,-0.032,
                0.054,-0.434,-0.173, 1.249,-0.167, 0.121, 0.032]))
print(list(quit_q))
print(quit_q)
quit_poseL, quit_poseR = FK(quit_q)
print("Quit Left hand pose:\n", quit_poseL)
print("Quit Right hand pose:\n", quit_poseR)

[0.053, 0.486, 0.173, 1.249, 0.167, -0.032, 0.054, -0.434, -0.173, 1.249, -0.167, 0.032]
[ 0.053  0.486  0.173  1.249  0.167 -0.032  0.054 -0.434 -0.173  1.249 -0.167  0.032]
Quit Left hand pose:
   R =
  0.262454 0.00779311   0.964913
  0.467737   0.873608  -0.134279
 -0.844002   0.486568   0.225636
  p = 0.0832375  0.397315 -0.150974

Quit Right hand pose:
   R =
   0.260962 -0.00527336    0.965335
  -0.424072    0.897704    0.119545
  -0.867215   -0.440568    0.232031
  p = 0.0824083 -0.374371  -0.16373

